In [1]:
# 02_baseline_y_ml.ipynb — setup inicial
import pandas as pd
import numpy as np
import json
from pathlib import Path

DATA_DIR = Path(r"E:\USUARIO\Desktop\Dataset 5")

# Cargar artefactos del notebook anterior
df_clean = pd.read_parquet(DATA_DIR / "Dataset_clean.parquet")
with open(DATA_DIR / "features_finales.json") as f:
    features_28 = json.load(f)

with np.load(DATA_DIR / "splits.npz") as npz:
    X_train = npz['X_train']
    X_test  = npz['X_test']
    y_train = npz['y_train']
    y_test  = npz['y_test']

print(f"Dataset: {df_clean.shape}")
print(f"Features: {len(features_28)}")
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

Dataset: (43984, 64)
Features: 28
Train: (30788, 28), Test: (13196, 28)


In [2]:
# ============================================================
# BASELINE Z-SCORE
# ============================================================
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score
import time

# Cargar los splits guardados
with np.load(DATA_DIR / "splits.npz") as npz:
    X_train = npz['X_train']
    X_test  = npz['X_test']
    y_train = npz['y_train']
    y_test  = npz['y_test']

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape:  {X_test.shape}")
print()

# --- PASO 1: Perfil de normalidad sobre el train normal ---
# Los datos ya están estandarizados (media 0, std 1 sobre el train completo),
# pero necesitamos el perfil sobre el train SOLO normal
# OJO: StandardScaler se aplicó sobre el train completo (normal + ataque),
# pero para el baseline Z-score necesitamos media/std de SOLO los normales.
# Recargamos y reescalamos:

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import json

with open(DATA_DIR / "features_finales.json") as f:
    features_28 = json.load(f)

X_raw = df_clean[features_28]
y     = df_clean['IT_B_Label']
X_train_raw, X_test_raw, y_train_raw, y_test_raw = train_test_split(
    X_raw, y, test_size=0.30, stratify=y, random_state=42
)

# Solo flujos normales del train
X_train_normal = X_train_raw[y_train_raw == 0]
print(f"Flujos normales en train: {len(X_train_normal):,}")

# Media y std sobre normales del train
mu    = X_train_normal.mean()
sigma = X_train_normal.std().replace(0, 1e-9)  # evitar div/0

# --- PASO 2: Z-score máximo para cada flujo del test ---
z_test = ((X_test_raw - mu).abs() / sigma).max(axis=1).values
print(f"\nEstadísticas del Z-score máximo en test:")
print(f"  min:     {z_test.min():.4f}")
print(f"  mediana: {np.median(z_test):.4f}")
print(f"  max:     {z_test.max():.4f}")

# --- PASO 3: Barrido de umbrales k ---
umbrales = [1.0, 1.5, 2.0, 2.5, 3.0, 4.0, 5.0]
t0 = time.time()
resultados_baseline = []
for k in umbrales:
    y_pred = (z_test > k).astype(int)
    prec = precision_score(y_test_raw, y_pred, zero_division=0)
    rec  = recall_score(y_test_raw, y_pred, zero_division=0)
    f1   = f1_score(y_test_raw, y_pred, zero_division=0)
    tp = int(((y_pred == 1) & (y_test_raw == 1)).sum())
    fp = int(((y_pred == 1) & (y_test_raw == 0)).sum())
    fn = int(((y_pred == 0) & (y_test_raw == 1)).sum())
    tn = int(((y_pred == 0) & (y_test_raw == 0)).sum())
    resultados_baseline.append({
        'k': k, 'Precision': round(prec, 4), 'Recall': round(rec, 4),
        'F1': round(f1, 4), 'TP': tp, 'FP': fp, 'FN': fn, 'TN': tn,
        'predichos_ataque': int(y_pred.sum())
    })

auc = roc_auc_score(y_test_raw, z_test)
tiempo = time.time() - t0

df_baseline = pd.DataFrame(resultados_baseline)
print(f"\n=== RESULTADOS BASELINE Z-SCORE ===")
print(df_baseline.to_string(index=False))
print(f"\nAUC-ROC (todo el barrido de umbrales): {auc:.4f}")
print(f"Tiempo total: {tiempo:.2f} s")

# Identificar el mejor k
mejor = df_baseline.loc[df_baseline['F1'].idxmax()]
print(f"\n=== MEJOR UMBRAL ===")
print(f"k = {mejor['k']}")
print(f"F1 = {mejor['F1']:.4f}")
print(f"Precision = {mejor['Precision']:.4f}")
print(f"Recall = {mejor['Recall']:.4f}")

# Guardar resultados para el capítulo 6
df_baseline.to_csv(DATA_DIR / "resultados_baseline.csv", index=False)
np.save(DATA_DIR / "baseline_scores.npy", z_test)
print(f"\nGuardado en: {DATA_DIR / 'resultados_baseline.csv'}")

X_train shape: (30788, 28)
X_test shape:  (13196, 28)

Flujos normales en train: 21,029

Estadísticas del Z-score máximo en test:
  min:     0.7422
  mediana: 1.5003
  max:     52000000000.0000

=== RESULTADOS BASELINE Z-SCORE ===
  k  Precision  Recall     F1   TP   FP   FN   TN  predichos_ataque
1.0     0.3086  0.9610 0.4671 4020 9008  163    5             13028
1.5     0.4242  0.6691 0.5192 2799 3800 1384 5213              6599
2.0     0.7500  0.5114 0.6081 2139  713 2044 8300              2852
2.5     0.7939  0.4889 0.6051 2045  531 2138 8482              2576
3.0     0.8258  0.4690 0.5983 1962  414 2221 8599              2376
4.0     0.8512  0.4432 0.5829 1854  324 2329 8689              2178
5.0     0.9203  0.3976 0.5553 1663  144 2520 8869              1807

AUC-ROC (todo el barrido de umbrales): 0.7137
Tiempo total: 0.15 s

=== MEJOR UMBRAL ===
k = 2.0
F1 = 0.6081
Precision = 0.7500
Recall = 0.5114

Guardado en: E:\USUARIO\Desktop\Dataset 5\resultados_baseline.csv


In [5]:
# Verificación del setup del notebook 02
print(f"df_clean:     {df_clean.shape}")
print(f"features_28:  {len(features_28)} features")
print(f"X_train:      {X_train.shape}")
print(f"X_test:       {X_test.shape}")
print(f"y_train:      {y_train.shape} | ataques: {int((y_train==1).sum()):,}")
print(f"y_test:       {y_test.shape} | ataques: {int((y_test==1).sum()):,}")

# Comprobamos que el baseline también se cargó bien
try:
    df_baseline = pd.read_csv(DATA_DIR / "resultados_baseline.csv")
    print(f"\nBaseline cargado OK: {len(df_baseline)} filas")
    print(f"Mejor F1 del baseline: {df_baseline['F1'].max():.4f}")
except Exception as e:
    print(f"\nError cargando baseline: {e}")

# Verificamos librerías que vamos a necesitar para los 4 modelos ML
print("\n=== LIBRERÍAS NECESARIAS ===")
try:
    import sklearn
    print(f"scikit-learn: {sklearn.__version__}")
except ImportError:
    print("scikit-learn NO instalado")

try:
    import xgboost
    print(f"xgboost:      {xgboost.__version__}")
except ImportError:
    print("xgboost NO instalado")

try:
    import joblib
    print(f"joblib:       {joblib.__version__}")
except ImportError:
    print("joblib NO instalado")

df_clean:     (43984, 64)
features_28:  28 features
X_train:      (30788, 28)
X_test:       (13196, 28)
y_train:      (30788,) | ataques: 9,759
y_test:       (13196,) | ataques: 4,183

Baseline cargado OK: 7 filas
Mejor F1 del baseline: 0.6081

=== LIBRERÍAS NECESARIAS ===
scikit-learn: 1.5.1
xgboost:      3.2.0
joblib:       1.4.2


In [6]:
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (precision_score, recall_score, f1_score,
                              roc_auc_score, confusion_matrix, classification_report)
import joblib
import time

# Carpetas de salida
MODELS_DIR  = DATA_DIR / "models"
RESULTS_DIR = DATA_DIR / "results"
MODELS_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)

# Acumulador de métricas
metricas_ml = []

def resumen_modelo(nombre, y_true, y_pred, y_scores, tiempo_train):
    """Calcula y muestra las métricas estándar de un modelo."""
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec  = recall_score(y_true, y_pred, zero_division=0)
    f1   = f1_score(y_true, y_pred, zero_division=0)
    auc  = roc_auc_score(y_true, y_scores) if y_scores is not None else np.nan
    cm   = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    print(f"\n=== {nombre} ===")
    print(f"Tiempo entrenamiento: {tiempo_train:.2f} s")
    print(f"Precision: {prec:.4f}")
    print(f"Recall:    {rec:.4f}")
    print(f"F1:        {f1:.4f}")
    print(f"AUC-ROC:   {auc:.4f}")
    print(f"Matriz de confusión:")
    print(f"          pred=0    pred=1")
    print(f"real=0    {tn:6d}    {fp:6d}")
    print(f"real=1    {fn:6d}    {tp:6d}")
    return {
        'modelo': nombre, 'Precision': round(prec, 4), 'Recall': round(rec, 4),
        'F1': round(f1, 4), 'AUC_ROC': round(auc, 4),
        'TP': int(tp), 'FP': int(fp), 'FN': int(fn), 'TN': int(tn),
        'tiempo_s': round(tiempo_train, 2)
    }

In [7]:
# --- RANDOM FOREST ---
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=20,
    class_weight='balanced',
    n_jobs=-1,
    random_state=42
)

t0 = time.time()
rf.fit(X_train, y_train)
tiempo_rf = time.time() - t0

# Predicciones
y_pred_rf   = rf.predict(X_test)
y_scores_rf = rf.predict_proba(X_test)[:, 1]

# Métricas
m_rf = resumen_modelo("Random Forest", y_test, y_pred_rf, y_scores_rf, tiempo_rf)
metricas_ml.append(m_rf)

# Guardar modelo, predicciones, importancia de features
joblib.dump(rf, MODELS_DIR / "rf.joblib")
np.save(RESULTS_DIR / "preds_rf.npy", y_pred_rf)
np.save(RESULTS_DIR / "scores_rf.npy", y_scores_rf)

# Importancia de variables
importancia_rf = pd.DataFrame({
    'feature': features_28,
    'importancia': rf.feature_importances_
}).sort_values('importancia', ascending=False).reset_index(drop=True)
importancia_rf.to_csv(RESULTS_DIR / "importancia_rf.csv", index=False)

print("\n=== TOP 10 FEATURES MÁS IMPORTANTES (Random Forest) ===")
print(importancia_rf.head(10).to_string(index=False))
print(f"\nModelo guardado en: {MODELS_DIR / 'rf.joblib'}")


=== Random Forest ===
Tiempo entrenamiento: 4.84 s
Precision: 0.8731
Recall:    0.6758
F1:        0.7619
AUC-ROC:   0.8771
Matriz de confusión:
          pred=0    pred=1
real=0      8602       411
real=1      1356      2827

=== TOP 10 FEATURES MÁS IMPORTANTES (Random Forest) ===
        feature  importancia
   sAckDelayAvg     0.164891
rInterPacketAvg     0.102915
   rAckDelayMax     0.092980
       duration     0.083539
sInterPacketAvg     0.082099
       sPackets     0.080591
          rLoad     0.072064
          sLoad     0.070853
    sPayloadSum     0.070463
   sAckDelayMax     0.038068

Modelo guardado en: E:\USUARIO\Desktop\Dataset 5\models\rf.joblib


In [8]:
# --- XGBOOST ---
xgb = XGBClassifier(
    n_estimators=200,
    max_depth=8,
    learning_rate=0.1,
    scale_pos_weight=2.15,
    n_jobs=-1,
    random_state=42,
    eval_metric='logloss'
)

t0 = time.time()
xgb.fit(X_train, y_train)
tiempo_xgb = time.time() - t0

# Predicciones
y_pred_xgb   = xgb.predict(X_test)
y_scores_xgb = xgb.predict_proba(X_test)[:, 1]

# Métricas
m_xgb = resumen_modelo("XGBoost", y_test, y_pred_xgb, y_scores_xgb, tiempo_xgb)
metricas_ml.append(m_xgb)

# Guardar
joblib.dump(xgb, MODELS_DIR / "xgb.joblib")
np.save(RESULTS_DIR / "preds_xgb.npy", y_pred_xgb)
np.save(RESULTS_DIR / "scores_xgb.npy", y_scores_xgb)

# Importancia de variables
importancia_xgb = pd.DataFrame({
    'feature': features_28,
    'importancia': xgb.feature_importances_
}).sort_values('importancia', ascending=False).reset_index(drop=True)
importancia_xgb.to_csv(RESULTS_DIR / "importancia_xgb.csv", index=False)

print("\n=== TOP 10 FEATURES MÁS IMPORTANTES (XGBoost) ===")
print(importancia_xgb.head(10).to_string(index=False))
print(f"\nModelo guardado en: {MODELS_DIR / 'xgb.joblib'}")


=== XGBoost ===
Tiempo entrenamiento: 1.11 s
Precision: 0.8207
Recall:    0.7169
F1:        0.7653
AUC-ROC:   0.8803
Matriz de confusión:
          pred=0    pred=1
real=0      8358       655
real=1      1184      2999

=== TOP 10 FEATURES MÁS IMPORTANTES (XGBoost) ===
        feature  importancia
       sPackets     0.303729
           sttl     0.194568
       sPshRate     0.144723
      sBytesMin     0.058188
      sBytesMax     0.041220
   sAckDelayAvg     0.037232
sInterPacketAvg     0.037150
      rBytesMin     0.029386
    rPayloadMax     0.028059
   rAckDelayMax     0.026552

Modelo guardado en: E:\USUARIO\Desktop\Dataset 5\models\xgb.joblib


In [9]:
# Tabla comparativa de los dos modelos + baseline
df_metricas = pd.DataFrame(metricas_ml)

# Añadir el mejor baseline como referencia
baseline_row = pd.DataFrame([{
    'modelo': 'Baseline (k=2)',
    'Precision': 0.7500, 'Recall': 0.5114, 'F1': 0.6081,
    'AUC_ROC': 0.7137,
    'TP': 2139, 'FP': 713, 'FN': 2044, 'TN': 8300,
    'tiempo_s': 0.13
}])

df_comparativa = pd.concat([baseline_row, df_metricas], ignore_index=True)
df_comparativa.to_csv(RESULTS_DIR / "metricas_modelos.csv", index=False)

print("=== COMPARATIVA BASELINE vs ML SUPERVISADOS ===")
print(df_comparativa.to_string(index=False))

=== COMPARATIVA BASELINE vs ML SUPERVISADOS ===
        modelo  Precision  Recall     F1  AUC_ROC   TP  FP   FN   TN  tiempo_s
Baseline (k=2)     0.7500  0.5114 0.6081   0.7137 2139 713 2044 8300      0.13
 Random Forest     0.8731  0.6758 0.7619   0.8771 2827 411 1356 8602      4.84
       XGBoost     0.8207  0.7169 0.7653   0.8803 2999 655 1184 8358      1.11


In [10]:
from sklearn.ensemble import IsolationForest

# Entrenamiento SOLO con flujos normales del train
X_train_normal = X_train[y_train == 0]
print(f"Flujos normales para entrenar IF y LOF: {len(X_train_normal):,}")
print()

# --- ISOLATION FOREST ---
iforest = IsolationForest(
    n_estimators=200,
    contamination=0.05,
    n_jobs=-1,
    random_state=42
)

t0 = time.time()
iforest.fit(X_train_normal)
tiempo_if = time.time() - t0

# Predicción: IF devuelve +1 (normal) y -1 (anomalía). Convertimos a 0/1.
y_pred_if_raw = iforest.predict(X_test)
y_pred_if = (y_pred_if_raw == -1).astype(int)

# Score de anomalía (más negativo = más anómalo). Invertimos signo.
y_scores_if = -iforest.score_samples(X_test)

# Métricas
m_if = resumen_modelo("Isolation Forest", y_test, y_pred_if, y_scores_if, tiempo_if)
metricas_ml.append(m_if)

# Guardar
joblib.dump(iforest, MODELS_DIR / "iforest.joblib")
np.save(RESULTS_DIR / "preds_iforest.npy", y_pred_if)
np.save(RESULTS_DIR / "scores_iforest.npy", y_scores_if)
print(f"\nModelo guardado en: {MODELS_DIR / 'iforest.joblib'}")

Flujos normales para entrenar IF y LOF: 21,029


=== Isolation Forest ===
Tiempo entrenamiento: 0.86 s
Precision: 0.7882
Recall:    0.3727
F1:        0.5061
AUC-ROC:   0.7431
Matriz de confusión:
          pred=0    pred=1
real=0      8594       419
real=1      2624      1559

Modelo guardado en: E:\USUARIO\Desktop\Dataset 5\models\iforest.joblib


In [11]:
from sklearn.neighbors import LocalOutlierFactor

# --- LOCAL OUTLIER FACTOR (LOF) ---
# novelty=True permite ajustar solo con los normales del train y predecir sobre test no visto,
# manteniendo la misma separación train/test que los modelos supervisados.
lof = LocalOutlierFactor(
    n_neighbors=20,
    contamination=0.05,
    novelty=True,
    n_jobs=-1
)

t0 = time.time()
lof.fit(X_train_normal)
tiempo_lof = time.time() - t0

# Predicción: LOF devuelve +1 (normal) y -1 (anomalía). Convertimos a 0/1.
y_pred_lof_raw = lof.predict(X_test)
y_pred_lof = (y_pred_lof_raw == -1).astype(int)

# Score de anomalía: score_samples devuelve "más alto = más normal". Invertimos signo.
y_scores_lof = -lof.score_samples(X_test)

# Métricas
m_lof = resumen_modelo("LOF", y_test, y_pred_lof, y_scores_lof, tiempo_lof)
metricas_ml.append(m_lof)

# Guardar
joblib.dump(lof, MODELS_DIR / "lof.joblib")
np.save(RESULTS_DIR / "preds_lof.npy", y_pred_lof)
np.save(RESULTS_DIR / "scores_lof.npy", y_scores_lof)
print(f"\nModelo guardado en: {MODELS_DIR / 'lof.joblib'}")


=== LOF ===
Tiempo entrenamiento: 1.40 s
Precision: 0.8395
Recall:    0.5965
F1:        0.6974
AUC-ROC:   0.8060
Matriz de confusión:
          pred=0    pred=1
real=0      8536       477
real=1      1688      2495

Modelo guardado en: E:\USUARIO\Desktop\Dataset 5\models\lof.joblib
